# Chess Human Eval - Training on Google Colab

Full pipeline: Download Lichess data -> Extract -> Encode -> Train

**Runtime:** T4 GPU (free tier works)

## 1. Setup

In [2]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/ChessNeuralNetwork'
!mkdir -p {DRIVE_DIR}/checkpoints {DRIVE_DIR}/data/chunks

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Clone repo
!git clone https://github.com/Clappedbytxger/chess-human-eval.git /content/chess
%cd /content/chess

In [ ]:
# Install dependencies
!pip install -q python-chess h5py pandas pyarrow zstandard tqdm tensorboard

In [ ]:
# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Data Pipeline

Downloads a Lichess monthly archive, filters Rapid/Classical, extracts training samples, builds HDF5 chunks.

**If you already have chunks on Drive**, skip to Section 3.

In [ ]:
# Configuration
YEAR = 2024
MONTH = 1            # Pick a month
MAX_GAMES = 200_000  # Limit games (None = all, but can take hours)
CHUNK_SIZE = 100_000 # Positions per HDF5 chunk

In [ ]:
# Step 1: Download PGN archive
from data.download import download_pgn
from pathlib import Path

pgn_path = download_pgn(YEAR, MONTH, output_dir=Path('/content/data/raw'))
print(f"Downloaded: {pgn_path}")
print(f"Size: {pgn_path.stat().st_size / 1e9:.2f} GB")

In [ ]:
# Step 2: Extract samples (FEN, move, Elo) from filtered games
from data.extract_samples import extract_samples_to_parquet

output_path = Path('/content/data/processed/samples.parquet')
output_path.parent.mkdir(parents=True, exist_ok=True)

parquet_path = extract_samples_to_parquet(
    pgn_path,
    output_path=output_path,
    max_games=MAX_GAMES,
)

import pandas as pd
df = pd.read_parquet(parquet_path)
print(f"Total samples: {len(df):,}")
print(f"Elo range: {df['active_elo'].min()} - {df['active_elo'].max()}")
print(df['active_elo'].describe())

In [ ]:
# Step 3: Build HDF5 training chunks
from data.build_dataset import build_chunks

chunks_dir = Path(DRIVE_DIR) / 'data' / 'chunks'
chunk_paths = build_chunks(
    parquet_path,
    output_dir=chunks_dir,
    chunk_size=CHUNK_SIZE,
)
print(f"Created {len(chunk_paths)} chunks in {chunks_dir}")

In [ ]:
# Verify chunks
import h5py

chunks_dir = Path(DRIVE_DIR) / 'data' / 'chunks'
total_samples = 0
for cp in sorted(chunks_dir.glob('chunk_*.h5')):
    with h5py.File(cp, 'r') as f:
        n = len(f['boards'])
        total_samples += n
        print(f"{cp.name}: {n:,} samples")

print(f"\nTotal: {total_samples:,} training samples")

## 3. Training

In [6]:
import os
print("DRIVE_DIR:", DRIVE_DIR)
print("Contents:", os.listdir(DRIVE_DIR))
print("training/ exists:", os.path.isdir(os.path.join(DRIVE_DIR, 'training')))

DRIVE_DIR: /content/drive/MyDrive/ChessNeuralNetwork
Contents: ['checkpoints', 'data']
training/ exists: False


In [3]:
from pathlib import Path
!git clone https://github.com/Clappedbytxger/chess-human-eval.git /content/repo
import sys
sys.path.insert(0, '/content/repo')
from training.config import TrainConfig
!pip install -r /content/repo/requirements.txt
from training.train import train

CHUNKS_DIR = Path(DRIVE_DIR) / 'data' / 'chunks'
CHECKPOINT_DIR = Path(DRIVE_DIR) / 'checkpoints'

config = TrainConfig(
    chunks_dir=CHUNKS_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    tensorboard_dir=Path('/content/runs'),

    # Model
    num_channels=128,
    num_blocks=10,
    embed_dim=32,

    # Training
    batch_size=128,
    learning_rate=0.01,
    momentum=0.9,
    weight_decay=1e-4,
    max_grad_norm=1.0,
    num_epochs=5,
    value_loss_weight=0.5,

    # Schedule
    use_cosine_annealing=True,

    # Checkpointing (every 30 min to Drive)
    checkpoint_interval_min=30,

    # Logging
    log_interval=50,

    # Hardware
    device='auto',
    num_workers=0,
)

print(f"Chunks: {config.chunks_dir}")
print(f"Checkpoints: {config.checkpoint_dir}")
print(f"Device: {config.resolve_device()}")

fatal: destination path '/content/repo' already exists and is not an empty directory.
Chunks: /content/drive/MyDrive/ChessNeuralNetwork/data/chunks
Checkpoints: /content/drive/MyDrive/ChessNeuralNetwork/checkpoints
Device: cuda


In [ ]:
# Start training (fresh)
train(config)

Training on: cuda
Model parameters: 3,886,730


In [ ]:
# Resume from checkpoint (if Colab disconnected)
# Uncomment and adjust the checkpoint path:
# train(config, resume_from=CHECKPOINT_DIR / 'epoch_1.pt')

In [ ]:
# TensorBoard
%load_ext tensorboard
%tensorboard --logdir /content/runs

## 4. Quick Evaluation

In [ ]:
import torch
import chess
from model.chess_net import ChessNet
from model.board_encoder import BoardEncoder
from model.policy_head import get_legal_move_mask, policy_index_to_move
from training.checkpoint import CheckpointManager

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ChessNet().to(device)

ckpt_path = Path(DRIVE_DIR) / 'checkpoints' / 'best_model.pt'
if not ckpt_path.exists():
    ckpt_path = Path(DRIVE_DIR) / 'checkpoints' / 'final_model.pt'
CheckpointManager.load(ckpt_path, model, device=device)
model.eval()

def predict_moves(fen, elo, top_k=5):
    board = chess.Board(fen)
    encoder = BoardEncoder()
    board_t = torch.from_numpy(encoder.encode_board(board)).unsqueeze(0).to(device)
    elo_t = torch.tensor([elo], dtype=torch.float32).to(device)
    mask = torch.from_numpy(get_legal_move_mask(board)).unsqueeze(0).to(device)

    with torch.no_grad():
        logprobs, value = model(board_t, elo_t, mask)

    probs = torch.exp(logprobs[0])
    top_vals, top_idx = probs.topk(top_k)

    print(f"Elo {elo} | Value: {value.item():.3f}")
    for p, i in zip(top_vals.cpu().numpy(), top_idx.cpu().numpy()):
        move = policy_index_to_move(int(i), board)
        if move and move in board.legal_moves:
            print(f"  {board.san(move):<8} {p:>6.1%}")

# Starting position at different Elos
fen = 'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1'
for elo in [1000, 1500, 2000, 2500]:
    predict_moves(fen, elo)
    print()

## 5. Download Checkpoint

In [ ]:
# List checkpoints
ckpt_dir = Path(DRIVE_DIR) / 'checkpoints'
for f in sorted(ckpt_dir.glob('*.pt')):
    print(f"{f.name}: {f.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Download to local machine
from google.colab import files
files.download(str(ckpt_dir / 'best_model.pt'))